Подготовка среда

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
!pip list 2>/dev/null | grep -i -E "^(torch|diffusers|transformers|pillow)"

diffusers                             0.41.0
pillow                                11.3.0
torch                                 2.11.0+cu130
torchao                               0.10.0
torchaudio                            2.11.0+cu130
torchcodec                            0.11.0+cu130
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cu130
transformers                          5.20.2


In [ ]:
%%writefile lab05_lib.py

# ЛР 5. Запуск открытой модели text-to-image. Библиотека учебных функций.
# Проверено 19.09.2026: Google Colab, GPU T4, Python 3.13, torch 2.11.0+cu128, diffusers 0.40.0, transformers 5.16.1.
# Модель: stabilityai/sd-turbo, ревизия закреплена (см. REVISION). Веса загружаются с Hugging Face без авторизации.
import hashlib, io, time
import numpy as np
import torch
from PIL import Image
from diffusers import AutoPipelineForText2Image
from diffusers.utils import logging as dlog

dlog.set_verbosity_error()  # убирает предупреждения о приведении типов при pipe.to(dtype)

MODEL_ID = "stabilityai/sd-turbo"
REVISION = "b261bac6fd2cf515557d5d0707481eafa0485ec2"

# Пять контекстов (брифов). Во всех запретены текст, логотипы и люди.
BRIEFS = [
    "editorial poster concept for a robotics festival, glowing mechanical arm silhouette, deep blue and amber palette, clean composition, no text, no logo, no people",
    "square podcast cover about the city environment, abstract layered skyline of geometric shapes, teal and orange palette, minimal flat style, no text, no people",
    "poster concept for a science exhibition, abstract constellation of connected nodes, dark background with white and cyan lines, balanced composition, no text, no logo",
    "visual for an ecology lecture series, stylized forest canopy seen from above, layered green shapes, soft morning light, no text, no people",
    "cover art for an electronic music release, brushed metal surface with iridescent reflections, macro photography style, strong contrast, no text, no logo, no people",
]


def load_pipe(dtype=torch.float16):
    """Загружает закреплённую ревизию (файлы fp16) на GPU; dtype задаёт тип вычислений."""
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID, revision=REVISION, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    pipe.set_progress_bar_config(disable=True)
    return pipe.to("cuda", dtype)


def generate(pipe, prompt, seed, steps=1, guidance=0.0, size=512, batch=1, gen_device="cpu"):
    """Возвращает список PIL-изображений. Для каждого элемента пакета свой генератор: seed, seed+1, ..."""
    gens = [torch.Generator(device=gen_device).manual_seed(seed + i) for i in range(batch)]
    return pipe(prompt=[prompt] * batch, num_inference_steps=steps, guidance_scale=guidance,
                height=size, width=size, generator=gens).images


def generate_fixed_noise(pipe, prompt, seed, steps=1, guidance=0.0, size=512):
    """Начальный шум всегда строится в float32 и приводится к типу вычислений: так меняется только точность."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    lat = torch.randn((1, 4, size // 8, size // 8), generator=g, dtype=torch.float32).to("cuda", pipe.unet.dtype)
    return pipe(prompt=[prompt], num_inference_steps=steps, guidance_scale=guidance, height=size, width=size, latents=lat).images[0]


def png_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def sha(data):
    return hashlib.sha256(data).hexdigest()


def to_arr(img):
    return np.asarray(img.convert("RGB"), dtype=np.float64)


def psnr_db(a, b):
    """PSNR для значений 0..255; inf, если изображения совпадают."""
    mse = np.mean((to_arr(a) - to_arr(b)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def max_abs_diff(a, b):
    return int(np.max(np.abs(to_arr(a) - to_arr(b))))


def jpeg_roundtrip(img, quality=90):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB"), buf.getvalue()


def classify(psnr, tol_db=40.0):
    """Классификация пары по допуску, заданному до эксперимента."""
    return "identical" if psnr == float("inf") else ("close" if psnr >= tol_db else "different")


# Шесть условий сравнения (фактор варианта). Каждое условие даёт пару изображений A и B.
CONDITIONS = ["repeat", "seed_plus_1", "generator_device", "dtype", "file_format", "batch"]


def make_pair(pipe, prompt, seed, cond):
    if cond == "repeat":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed)[0]
    elif cond == "seed_plus_1":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed + 1)[0]
    elif cond == "generator_device":
        a = generate(pipe, prompt, seed, gen_device="cpu")[0]; b = generate(pipe, prompt, seed, gen_device="cuda")[0]
    elif cond == "dtype":
        pipe.to(torch.float16); a = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float32); b = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float16)
    elif cond == "file_format":
        a = generate(pipe, prompt, seed)[0]; b, _ = jpeg_roundtrip(a, 90)
    elif cond == "batch":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed, batch=4)[0]
    else:
        raise ValueError(cond)
    return a, b


def variant_summary(pipe, n):
    """Вариант n = 1..30: контекст c = (n-1)//6 + 1, условие k = (n-1)%6 + 1."""
    c, k = (n - 1) // 6, (n - 1) % 6
    seed = 1000 + n
    t0 = time.perf_counter()
    a, b = make_pair(pipe, BRIEFS[c], seed, CONDITIONS[k])
    dt = time.perf_counter() - t0
    p = psnr_db(a, b)
    pf = None
    if CONDITIONS[k] == "dtype":  # проверка причины: одинаковый начальный шум в обоих режимах
        pipe.to(torch.float16); fa = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float32); fb = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float16); pf = psnr_db(fa, fb)
    return dict(variant=n, ctx=c + 1, cond=CONDITIONS[k], seed=seed, sha_a=sha(png_bytes(a)), sha_b=sha(png_bytes(b)),
                pixel_equal=bool(np.array_equal(to_arr(a), to_arr(b))), max_diff=max_abs_diff(a, b), psnr=p,
                verdict=classify(p), seconds=round(dt, 2), psnr_fixed_noise=pf)


Writing lab05_lib.py


In [ ]:
!pip install diffusers==0.39.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 22.0 MB/s  0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.41.0
    Uninstalling diffusers-0.41.0:
      Successfully uninstalled diffusers-0.41.0


Загрузка модели и прогрев

In [ ]:
import json, pathlib, platform, sys, time
import torch, diffusers, transformers, PIL
import lab05_lib as L

BRIEF_IDX = 2          # бриф 3 (0-индексация)
SEED = 1015            # 1000 + 15
STEPS, GUIDANCE, SIZE = 1, 0.0, 512

t0 = time.perf_counter()
pipe = L.load_pipe()
print("load_seconds:", round(time.perf_counter()-t0, 1))
_ = L.generate(pipe, L.BRIEFS[BRIEF_IDX], SEED)   # прогрев
print("прогрев выполнен")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

load_seconds: 19.6
прогрев выполнен


Базовый запуск + манифест

In [ ]:
out = pathlib.Path("artifacts/run_015"); out.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
img = L.generate(pipe, L.BRIEFS[BRIEF_IDX], SEED, STEPS, GUIDANCE, SIZE)[0]
gen_s = time.perf_counter() - t0

data = L.png_bytes(img)
(out / "result.png").write_bytes(data)

manifest = dict(
    variant=15, brief=3, condition="generator_device",
    model_id=L.MODEL_ID, revision=L.REVISION,
    prompt=L.BRIEFS[BRIEF_IDX], seed=SEED, steps=STEPS,
    guidance_scale=GUIDANCE, size=SIZE,
    generator_device="cpu", dtype="float16",
    device=torch.cuda.get_device_name(0),
    platform=platform.platform(),
    python=sys.version.split()[0],
    packages={"torch": torch.__version__, "diffusers": diffusers.__version__,
              "transformers": transformers.__version__, "pillow": PIL.__version__},
    generation_seconds=round(gen_s, 2),
    image_size=list(img.size), image_mode=img.mode,
    sha256=L.sha(data),
)
(out / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "variant": 15,
  "brief": 3,
  "condition": "generator_device",
  "model_id": "stabilityai/sd-turbo",
  "revision": "b261bac6fd2cf515557d5d0707481eafa0485ec2",
  "prompt": "poster concept for a science exhibition, abstract constellation of connected nodes, dark background with white and cyan lines, balanced composition, no text, no logo",
  "seed": 1015,
  "steps": 1,
  "guidance_scale": 0.0,
  "size": 512,
  "generator_device": "cpu",
  "dtype": "float16",
  "device": "Tesla T4",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.16",
  "packages": {
    "torch": "2.11.0+cu130",
    "diffusers": "0.39.0",
    "transformers": "5.20.2",
    "pillow": "11.3.0"
  },
  "generation_seconds": 0.34,
  "image_size": [
    512,
    512
  ],
  "image_mode": "RGB",
  "sha256": "bd07f369e95e6616d677b5f69590fb5363fadcb295efc4c55b37b2c683a776c3"
}


Эксперимент варианта (GPU + CPU)

In [ ]:
summary = L.variant_summary(pipe, 15)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "variant": 15,
  "ctx": 3,
  "cond": "generator_device",
  "seed": 1015,
  "sha_a": "bd07f369e95e6616d677b5f69590fb5363fadcb295efc4c55b37b2c683a776c3",
  "sha_b": "0e30c7222c63d3151aab7e6aba3acfe77db5173de5c619b57b42b2c23cdbf11c",
  "pixel_equal": false,
  "max_diff": 248,
  "psnr": 11.62218514732884,
  "verdict": "different",
  "seconds": 0.64,
  "psnr_fixed_noise": null
}


Дополнительная проверка причины

In [ ]:
# A: генератор на CPU
a = L.generate(pipe, L.BRIEFS[BRIEF_IDX], SEED, gen_device="cpu")[0]
# B: генератор на GPU
b = L.generate(pipe, L.BRIEFS[BRIEF_IDX], SEED, gen_device="cuda")[0]

p = L.psnr_db(a, b)
print("SHA_a:", L.sha(L.png_bytes(a)))
print("SHA_b:", L.sha(L.png_bytes(b)))
print("pixel_equal:", (L.to_arr(a) == L.to_arr(b)).all())
print("max_abs_diff:", L.max_abs_diff(a, b))
print("PSNR:", round(p, 2), "dB")
print("verdict:", L.classify(p))

SHA_a: bd07f369e95e6616d677b5f69590fb5363fadcb295efc4c55b37b2c683a776c3
SHA_b: 0e30c7222c63d3151aab7e6aba3acfe77db5173de5c619b57b42b2c23cdbf11c
pixel_equal: False
max_abs_diff: 248
PSNR: 11.62 dB
verdict: different


Сохранение результата

In [ ]:
a.save(out / "A_cpu_gen.png")
b.save(out / "B_cuda_gen.png")

Намеренно введенная ошибка

In [ ]:
g = torch.Generator(device="cpu").manual_seed(SEED)
e1 = pipe(prompt=L.BRIEFS[BRIEF_IDX], num_inference_steps=STEPS,
          guidance_scale=GUIDANCE, generator=g).images[0]
e2 = pipe(prompt=L.BRIEFS[BRIEF_IDX], num_inference_steps=STEPS,
          guidance_scale=GUIDANCE, generator=g).images[0]
print("e1 == эталон:", L.sha(L.png_bytes(e1)) == manifest["sha256"])
print("e2 == эталон:", L.sha(L.png_bytes(e2)) == manifest["sha256"])
print("PSNR(e1, e2):", round(L.psnr_db(e1, e2), 2), "dB")

e1 == эталон: True
e2 == эталон: False
PSNR(e1, e2): 11.56 dB
